# mass mapping decomposition

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Mass-mapping decomposition inside common support

This notebook examines where the simulation-dependent mass mapping enters after the
conditional discrepancy has been established inside common feature support. The main
calculation decomposes the logarithmic stellar-sum-to-M200c relation into two additive
stages: stellar mass to combined bound-subhalo mass, and combined bound-subhalo mass to
M200c.

The analysis is evaluation-only. It does not fit, update or select a neural posterior
estimator. It uses the fixed held-out populations and out-of-fold domain scores from the
completed common-support analysis. The aim is to localise the discrepancy, not to assign
physical correctness to either simulation.


## 1. Estimands and interpretation

For each pair, define

\[
\Delta_{\rm star\rightarrow bound}
= \log_{10}(M_{\rm bound,1}+M_{\rm bound,2})
- \log_{10}(M_{\star,1}+M_{\star,2}),
\]

\[
\Delta_{\rm bound\rightarrow 200c}
= \log_{10}(M_{200c})
- \log_{10}(M_{\rm bound,1}+M_{\rm bound,2}),
\]

so that the row-level identity

\[
\Delta_{\rm star\rightarrow 200c}
= \Delta_{\rm star\rightarrow bound}
+ \Delta_{\rm bound\rightarrow 200c}
\]

holds exactly. All three outcomes use the same selected rows and the same regression design.
Consequently, their fitted coefficients, adjusted simulation gaps and bootstrap draws must
also satisfy this identity up to numerical precision.

A positive TNG-minus-FLAMINGO gap in the first stage means more combined bound mass per unit
stellar mass in TNG at the same reference covariates. A positive gap in the second stage
means more M200c per unit combined bound mass. These are conditional catalogue-level
comparisons. They do not by themselves distinguish baryonic modelling, halo finding,
selection or other construction differences.


In [ ]:
RANDOM_SEED = 260928
BOOTSTRAP_REPEATS = 500
OVERLAP_HALF_WIDTHS = [0.10, 0.20, 0.50]
PRIMARY_HALF_WIDTH = 0.10
MIN_ROWS_PER_SIMULATION = 30
SPLINE_N_KNOTS = 5
SPLINE_DEGREE = 3
SPLINE_RIDGE_ALPHA = 1.0
CURVE_GRID_SIZE = 41
CURVE_REFERENCE_PER_SIMULATION = 2000
DOMAIN_RESULT_ID = "20260925T103101Z"
OUTPUT_ROOT = ""


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import subprocess
import sys
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import SplineTransformer


def find_project_root() -> Path:
    candidates = []
    if os.environ.get("LG_PROJECT"):
        candidates.append(Path(os.environ["LG_PROJECT"]))
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "src" / "lg_npe").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find the project. Set LG_PROJECT to the project path."
    )


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


PROJECT = find_project_root()
sys.path.insert(0, str(PROJECT / "src"))

domain_matches = list(
    (PROJECT / "out").glob(f"*/domain_common_support/{DOMAIN_RESULT_ID}")
)
if len(domain_matches) != 1:
    raise FileNotFoundError(
        "Expected exactly one fixed domain-common-support result for "
        f"{DOMAIN_RESULT_ID}; found {len(domain_matches)}."
    )
DOMAIN_RESULT = domain_matches[0]
DOMAIN_MANIFEST = DOMAIN_RESULT / "domain_overlap_manifest.json"

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_base = (
    Path(OUTPUT_ROOT).resolve()
    if OUTPUT_ROOT
    else PROJECT / "out" / "domain_shift" / "mass_mapping_decomposition"
)
OUTPUT = output_base / timestamp
OUTPUT.mkdir(parents=True, exist_ok=False)

try:
    git_revision = subprocess.run(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    git_revision = None

domain_scores = pd.read_csv(DOMAIN_RESULT / "domain_scores_oof.csv.gz")
posterior_join = pd.read_csv(
    DOMAIN_RESULT / "posterior_domain_join.csv.gz",
    usecols=["population", "pair_index"],
)
source_evaluation_pairs = np.sort(
    posterior_join.loc[
        posterior_join["population"] == "source_heldout", "pair_index"
    ].unique()
)
if len(source_evaluation_pairs) != 21_409:
    raise RuntimeError(
        f"Expected 21,409 held-out FLAMINGO pairs; found {len(source_evaluation_pairs):,}."
    )

versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": metadata.version("matplotlib"),
}
print(f"Project: {PROJECT}")
print(f"Fixed domain result: {DOMAIN_RESULT_ID}")
print(f"Output: {OUTPUT}")
print(json.dumps(versions, indent=2))


## 2. Fixed populations and aligned mass quantities

The source population is the same 21,409-pair held-out FLAMINGO set used in the posterior
evaluation. The transfer population contains all 1,530 corrected TNG pairs. The primary
decomposition uses the stellar-sum domain score because stellar sum is the common observed
mass input at the start of the chain.

The member-level sensitivity check uses the stellar-individual score. Each member's bound
mass is aligned with the stellar-primary or stellar-secondary endpoint using the stored
endpoint flag. Catalogue endpoint order is therefore never treated as stellar-mass order.


In [ ]:
from lg_npe.data import load_flamingo_pair_table, load_tng_pair_table
from lg_npe.features import add_stellar_representations


MAIN_OUTCOMES = OrderedDict([
    ("stellar_to_bound", "log_bound_subhalo_sum_minus_log_stellar_sum"),
    ("bound_to_m200c", "log_m200c_minus_log_bound_subhalo_sum"),
    ("stellar_to_m200c", "log_m200c_minus_log_stellar_sum"),
])
MEMBER_OUTCOMES = OrderedDict([
    ("primary_bound_per_stellar", "log_bound_primary_minus_log_mstar_primary"),
    ("secondary_bound_per_stellar", "log_bound_secondary_minus_log_mstar_secondary"),
])
ANALYSES = OrderedDict([
    ("pair_sum_decomposition", {
        "feature_space": "stellar_sum",
        "score_column": "domain_score_stellar_sum",
        "component_column": "connected_component_stellar_sum",
        "predictors": ["separation", "vr", "vt", "log_stellar_sum"],
        "outcomes": MAIN_OUTCOMES,
    }),
    ("member_alignment", {
        "feature_space": "stellar_individual",
        "score_column": "domain_score_stellar_individual",
        "component_column": "connected_component_stellar_individual",
        "predictors": [
            "separation", "vr", "vt",
            "log_mstar_primary", "log_mstar_secondary",
        ],
        "outcomes": MEMBER_OUTCOMES,
    }),
])


def prepare_pair_table(table, simulation: str) -> pd.DataFrame:
    frame = add_stellar_representations(table.frame)
    if simulation == "flamingo":
        frame = frame.loc[frame["pair_index"].isin(source_evaluation_pairs)].copy()
        if len(frame) != len(source_evaluation_pairs):
            raise RuntimeError("The held-out source population is incomplete.")
    else:
        frame = frame.copy()

    bound_a = frame["Msub_bound_1"].to_numpy(np.float64)
    bound_b = frame["Msub_bound_2"].to_numpy(np.float64)
    if np.any(bound_a <= 0) or np.any(bound_b <= 0):
        raise ValueError("Bound-subhalo masses must be strictly positive.")
    a_is_primary = frame["stellar_primary_is_a"].to_numpy(bool)
    frame["log_bound_primary"] = np.log10(
        np.where(a_is_primary, bound_a, bound_b)
    )
    frame["log_bound_secondary"] = np.log10(
        np.where(a_is_primary, bound_b, bound_a)
    )

    frame["log_bound_subhalo_sum_minus_log_stellar_sum"] = (
        frame["log_bound_subhalo_sum"] - frame["log_stellar_sum"]
    )
    frame["log_m200c_minus_log_bound_subhalo_sum"] = (
        frame["log_m200c"] - frame["log_bound_subhalo_sum"]
    )
    frame["log_m200c_minus_log_stellar_sum"] = (
        frame["log_m200c"] - frame["log_stellar_sum"]
    )
    frame["log_bound_primary_minus_log_mstar_primary"] = (
        frame["log_bound_primary"] - frame["log_mstar_primary"]
    )
    frame["log_bound_secondary_minus_log_mstar_secondary"] = (
        frame["log_bound_secondary"] - frame["log_mstar_secondary"]
    )

    identity_residual = (
        frame[MAIN_OUTCOMES["stellar_to_m200c"]]
        - frame[MAIN_OUTCOMES["stellar_to_bound"]]
        - frame[MAIN_OUTCOMES["bound_to_m200c"]]
    )
    if np.max(np.abs(identity_residual)) > 1e-10:
        raise RuntimeError("The mass-chain identity failed at row level.")

    for feature_space in ("stellar_sum", "stellar_individual"):
        suffix = feature_space
        score_columns = [
            "pair_index", "domain_score", "outer_fold", "connected_component"
        ]
        scores = domain_scores.loc[
            (domain_scores["feature_space"] == feature_space)
            & (domain_scores["simulation"] == simulation),
            score_columns,
        ].rename(columns={
            "domain_score": f"domain_score_{suffix}",
            "outer_fold": f"outer_fold_{suffix}",
            "connected_component": f"connected_component_{suffix}",
        })
        frame = frame.merge(scores, on="pair_index", how="left", validate="one_to_one")

    score_columns = [
        "domain_score_stellar_sum", "domain_score_stellar_individual",
        "connected_component_stellar_sum", "connected_component_stellar_individual",
    ]
    if frame[score_columns].isna().any().any():
        raise RuntimeError(f"Domain-score join failed for {simulation}.")

    required = sorted(set(
        ["pair_index", "same_host", *MAIN_OUTCOMES.values(), *MEMBER_OUTCOMES.values()]
        + [column for specification in ANALYSES.values() for column in specification["predictors"]]
        + score_columns
    ))
    finite_columns = [
        column for column in required
        if column != "same_host" and not column.startswith("connected_component_")
    ]
    finite = np.isfinite(frame[finite_columns].to_numpy(np.float64)).all(axis=1)
    if not finite.all():
        raise ValueError(
            f"The fixed {simulation} population has {int((~finite).sum())} invalid rows; "
            "do not silently change the population."
        )
    frame["simulation"] = simulation
    return frame.reset_index(drop=True)


source = prepare_pair_table(load_flamingo_pair_table(PROJECT), "flamingo")
transfer = prepare_pair_table(load_tng_pair_table(PROJECT), "tng")
frames = {"flamingo": source, "tng": transfer}

population_rows = []
for analysis, specification in ANALYSES.items():
    for simulation, frame in frames.items():
        population_rows.append({
            "analysis": analysis,
            "feature_space": specification["feature_space"],
            "simulation": simulation,
            "n": len(frame),
            "same_host": int(frame["same_host"].sum()),
            "different_host": int((~frame["same_host"]).sum()),
            "connected_components": int(
                frame[specification["component_column"]].nunique()
            ),
            "predictors": json.dumps(specification["predictors"]),
            "outcomes": json.dumps(specification["outcomes"]),
        })

population_registry = pd.DataFrame(population_rows)
population_registry.to_csv(OUTPUT / "decomposition_population_registry.csv", index=False)
population_registry


## 3. Linear conditional decomposition

Each simulation is fitted separately with the same centred and scaled design. Continuous
predictors are centred and scaled using an equal-simulation reference distribution. The
pooled design also contains the host-class indicator; same-host and different-host fits are
reported separately.

The adjusted gap is the TNG-minus-FLAMINGO intercept difference at the common reference
centre. Since the three pair-sum outcomes use one design matrix and a joint linear solve,
the reported total gap is exactly the sum of the two stage gaps apart from floating-point
rounding. Coefficients describe conditional associations, not causal effects.


In [ ]:
SUBSETS = OrderedDict([
    ("pooled", lambda frame: np.ones(len(frame), dtype=bool)),
    ("same_host", lambda frame: frame["same_host"].to_numpy(bool)),
    ("different_host", lambda frame: ~frame["same_host"].to_numpy(bool)),
])


def interval_label(half_width: float) -> str:
    if half_width >= 0.5:
        return "all"
    return f"{0.5 - half_width:.2f}-{0.5 + half_width:.2f}"


def select_population(
    frame: pd.DataFrame,
    specification: dict,
    subset: str,
    half_width: float,
) -> pd.DataFrame:
    subset_mask = SUBSETS[subset](frame)
    scores = frame[specification["score_column"]].to_numpy(np.float64)
    overlap_mask = np.abs(scores - 0.5) <= half_width + 1e-12
    return frame.loc[subset_mask & overlap_mask].copy()


def equal_reference_location_scale(
    source_frame: pd.DataFrame,
    transfer_frame: pd.DataFrame,
    columns: list[str],
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    source_values = source_frame[columns].to_numpy(np.float64)
    transfer_values = transfer_frame[columns].to_numpy(np.float64)
    centre = 0.5 * (source_values.mean(axis=0) + transfer_values.mean(axis=0))
    variance = 0.5 * (
        np.mean((source_values - centre) ** 2, axis=0)
        + np.mean((transfer_values - centre) ** 2, axis=0)
    )
    keep = np.isfinite(variance) & (variance > 1e-12)
    return centre[keep], np.sqrt(variance[keep]), [
        name for name, retained in zip(columns, keep) if retained
    ]


def make_linear_design(
    frame: pd.DataFrame,
    columns: list[str],
    centre: np.ndarray,
    scale: np.ndarray,
) -> np.ndarray:
    standardized = (frame[columns].to_numpy(np.float64) - centre) / scale
    return np.column_stack([np.ones(len(frame)), standardized])


def weighted_least_squares(
    design: np.ndarray,
    targets: np.ndarray,
    weights: np.ndarray | None = None,
) -> np.ndarray:
    if weights is None:
        return np.linalg.pinv(design.T @ design) @ (design.T @ targets)
    weighted_design = design * weights[:, None]
    return np.linalg.pinv(design.T @ weighted_design) @ (
        design.T @ (targets * weights[:, None])
    )


def prepare_linear_mapping(
    analysis: str,
    subset: str,
    half_width: float,
) -> dict | None:
    specification = ANALYSES[analysis]
    selected_source = select_population(source, specification, subset, half_width)
    selected_transfer = select_population(transfer, specification, subset, half_width)
    if min(len(selected_source), len(selected_transfer)) < MIN_ROWS_PER_SIMULATION:
        return None

    candidate_columns = specification["predictors"] + (
        ["same_host"] if subset == "pooled" else []
    )
    centre, scale, columns = equal_reference_location_scale(
        selected_source, selected_transfer, candidate_columns
    )
    outcome_names = list(specification["outcomes"])
    outcome_columns = list(specification["outcomes"].values())
    source_design = make_linear_design(selected_source, columns, centre, scale)
    transfer_design = make_linear_design(selected_transfer, columns, centre, scale)
    source_targets = selected_source[outcome_columns].to_numpy(np.float64)
    transfer_targets = selected_transfer[outcome_columns].to_numpy(np.float64)
    source_beta = weighted_least_squares(source_design, source_targets)
    transfer_beta = weighted_least_squares(transfer_design, transfer_targets)
    return {
        "source": selected_source,
        "transfer": selected_transfer,
        "columns": columns,
        "centre": centre,
        "scale": scale,
        "outcome_names": outcome_names,
        "outcome_columns": outcome_columns,
        "source_design": source_design,
        "transfer_design": transfer_design,
        "source_targets": source_targets,
        "transfer_targets": transfer_targets,
        "source_beta": source_beta,
        "transfer_beta": transfer_beta,
    }


linear_summary_rows = []
linear_coefficient_rows = []
identity_rows = []

for analysis, specification in ANALYSES.items():
    for subset in SUBSETS:
        for half_width in OVERLAP_HALF_WIDTHS:
            prepared = prepare_linear_mapping(analysis, subset, half_width)
            if prepared is None:
                linear_summary_rows.append({
                    "analysis": analysis,
                    "subset": subset,
                    "half_width": half_width,
                    "score_interval": interval_label(half_width),
                    "status": "insufficient_rows",
                })
                continue

            difference = prepared["transfer_beta"] - prepared["source_beta"]
            source_prediction = prepared["source_design"] @ prepared["source_beta"]
            transfer_prediction = prepared["transfer_design"] @ prepared["transfer_beta"]
            for outcome_index, outcome in enumerate(prepared["outcome_names"]):
                source_target = prepared["source_targets"][:, outcome_index]
                transfer_target = prepared["transfer_targets"][:, outcome_index]
                linear_summary_rows.append({
                    "analysis": analysis,
                    "feature_space": specification["feature_space"],
                    "outcome": outcome,
                    "subset": subset,
                    "half_width": half_width,
                    "score_interval": interval_label(half_width),
                    "status": "ok",
                    "n_flamingo": len(prepared["source"]),
                    "n_tng": len(prepared["transfer"]),
                    "components_flamingo": prepared["source"][
                        specification["component_column"]
                    ].nunique(),
                    "components_tng": prepared["transfer"][
                        specification["component_column"]
                    ].nunique(),
                    "raw_gap_tng_minus_flamingo": (
                        transfer_target.mean() - source_target.mean()
                    ),
                    "adjusted_gap_tng_minus_flamingo": difference[0, outcome_index],
                    "rmse_flamingo": np.sqrt(np.mean(
                        (source_prediction[:, outcome_index] - source_target) ** 2
                    )),
                    "rmse_tng": np.sqrt(np.mean(
                        (transfer_prediction[:, outcome_index] - transfer_target) ** 2
                    )),
                    "r2_flamingo": r2_score(
                        source_target, source_prediction[:, outcome_index]
                    ),
                    "r2_tng": r2_score(
                        transfer_target, transfer_prediction[:, outcome_index]
                    ),
                })
                for term_index, term in enumerate(["intercept", *prepared["columns"]]):
                    linear_coefficient_rows.append({
                        "analysis": analysis,
                        "feature_space": specification["feature_space"],
                        "outcome": outcome,
                        "subset": subset,
                        "half_width": half_width,
                        "score_interval": interval_label(half_width),
                        "term": term,
                        "coefficient_flamingo": prepared["source_beta"][term_index, outcome_index],
                        "coefficient_tng": prepared["transfer_beta"][term_index, outcome_index],
                        "difference_tng_minus_flamingo": difference[term_index, outcome_index],
                        "reference_centre": (
                            np.nan if term_index == 0 else prepared["centre"][term_index - 1]
                        ),
                        "reference_scale": (
                            np.nan if term_index == 0 else prepared["scale"][term_index - 1]
                        ),
                    })

            if analysis == "pair_sum_decomposition":
                outcome_index = {
                    name: index for index, name in enumerate(prepared["outcome_names"])
                }
                residual = (
                    difference[:, outcome_index["stellar_to_m200c"]]
                    - difference[:, outcome_index["stellar_to_bound"]]
                    - difference[:, outcome_index["bound_to_m200c"]]
                )
                for term, value in zip(["intercept", *prepared["columns"]], residual):
                    identity_rows.append({
                        "subset": subset,
                        "half_width": half_width,
                        "score_interval": interval_label(half_width),
                        "term": term,
                        "identity_residual": value,
                    })

linear_summary = pd.DataFrame(linear_summary_rows)
linear_coefficients = pd.DataFrame(linear_coefficient_rows)
identity_audit = pd.DataFrame(identity_rows)
if identity_audit["identity_residual"].abs().max() > 1e-9:
    raise RuntimeError("The fitted linear decomposition identity failed.")

linear_summary.to_csv(OUTPUT / "decomposition_linear_summary.csv", index=False)
linear_coefficients.to_csv(OUTPUT / "decomposition_linear_coefficients.csv", index=False)
identity_audit.to_csv(OUTPUT / "decomposition_identity_audit.csv", index=False)

linear_summary.loc[
    (linear_summary["half_width"] == PRIMARY_HALF_WIDTH)
    & (linear_summary["status"] == "ok"),
    [
        "analysis", "outcome", "subset", "n_flamingo", "n_tng",
        "raw_gap_tng_minus_flamingo", "adjusted_gap_tng_minus_flamingo",
        "rmse_flamingo", "rmse_tng",
    ],
].round(4).reset_index(drop=True)


## 4. Connected-host-component uncertainty

Connected host components are resampled with replacement within each simulation. A single
set of component weights is shared by all outcomes in an analysis and repeat, preserving
the covariance between stages and the exact additive identity. The selected rows, domain
scores, reference centre, reference scale and linear specification remain fixed.

These intervals quantify fitting uncertainty conditional on the selected catalogues and
model form. They do not include cosmic variance, uncertainty in catalogue construction,
uncertainty in the domain classifier or posterior retraining. Different-host estimates
remain exploratory when their retained TNG samples are small.


In [ ]:
def bootstrap_row_weights(groups: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    _, inverse = np.unique(np.asarray(groups), return_inverse=True)
    n_components = int(inverse.max()) + 1
    sampled = rng.integers(0, n_components, size=n_components)
    multiplicity = np.bincount(sampled, minlength=n_components).astype(np.float64)
    return multiplicity[inverse]


rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []
bootstrap_identity_residuals = []

for analysis_number, (analysis, specification) in enumerate(ANALYSES.items(), start=1):
    print(f"[{analysis_number}/{len(ANALYSES)}] {analysis}")
    for subset in SUBSETS:
        prepared = prepare_linear_mapping(analysis, subset, PRIMARY_HALF_WIDTH)
        if prepared is None:
            continue
        source_groups = prepared["source"][
            specification["component_column"]
        ].to_numpy()
        transfer_groups = prepared["transfer"][
            specification["component_column"]
        ].to_numpy()
        terms = ["intercept", *prepared["columns"]]
        outcome_index = {
            name: index for index, name in enumerate(prepared["outcome_names"])
        }

        for repeat in range(BOOTSTRAP_REPEATS):
            source_weights = bootstrap_row_weights(source_groups, rng)
            transfer_weights = bootstrap_row_weights(transfer_groups, rng)
            source_beta = weighted_least_squares(
                prepared["source_design"], prepared["source_targets"], source_weights
            )
            transfer_beta = weighted_least_squares(
                prepared["transfer_design"], prepared["transfer_targets"], transfer_weights
            )
            difference = transfer_beta - source_beta
            for outcome_number, outcome in enumerate(prepared["outcome_names"]):
                for term_number, term in enumerate(terms):
                    bootstrap_rows.append({
                        "analysis": analysis,
                        "feature_space": specification["feature_space"],
                        "outcome": outcome,
                        "subset": subset,
                        "half_width": PRIMARY_HALF_WIDTH,
                        "repeat": repeat,
                        "estimand": (
                            "adjusted_gap" if term == "intercept" else "slope_difference"
                        ),
                        "term": term,
                        "value": difference[term_number, outcome_number],
                    })

            if analysis == "pair_sum_decomposition":
                residual = (
                    difference[:, outcome_index["stellar_to_m200c"]]
                    - difference[:, outcome_index["stellar_to_bound"]]
                    - difference[:, outcome_index["bound_to_m200c"]]
                )
                bootstrap_identity_residuals.append(float(np.max(np.abs(residual))))

bootstrap_draws = pd.DataFrame(bootstrap_rows)
bootstrap_draws.to_csv(
    OUTPUT / "decomposition_component_bootstrap_draws.csv.gz", index=False
)
if max(bootstrap_identity_residuals) > 1e-8:
    raise RuntimeError("The bootstrap decomposition identity failed.")

bootstrap_summary = (
    bootstrap_draws.groupby(
        [
            "analysis", "feature_space", "outcome", "subset", "half_width",
            "estimand", "term",
        ],
        sort=False,
    )["value"]
    .agg(
        bootstrap_median="median",
        lower_95=lambda values: np.quantile(values, 0.025),
        upper_95=lambda values: np.quantile(values, 0.975),
        bootstrap_mean="mean",
        bootstrap_std="std",
        fraction_above_zero=lambda values: np.mean(values > 0),
        repeats="size",
    )
    .reset_index()
)
point_estimates = linear_coefficients.loc[
    linear_coefficients["half_width"] == PRIMARY_HALF_WIDTH,
    [
        "analysis", "feature_space", "outcome", "subset", "half_width",
        "term", "difference_tng_minus_flamingo",
    ],
].rename(columns={"difference_tng_minus_flamingo": "estimate"})
point_estimates["estimand"] = np.where(
    point_estimates["term"] == "intercept", "adjusted_gap", "slope_difference"
)
bootstrap_summary = bootstrap_summary.merge(
    point_estimates,
    on=[
        "analysis", "feature_space", "outcome", "subset", "half_width",
        "estimand", "term",
    ],
    how="left",
    validate="one_to_one",
)
if bootstrap_summary["estimate"].isna().any():
    raise RuntimeError("A bootstrap summary row is missing its point estimate.")
bootstrap_summary.to_csv(
    OUTPUT / "decomposition_component_bootstrap_summary.csv", index=False
)

bootstrap_summary.loc[
    bootstrap_summary["estimand"] == "adjusted_gap",
    [
        "analysis", "outcome", "subset", "estimate", "lower_95", "upper_95",
        "fraction_above_zero", "repeats",
    ],
].round(4).reset_index(drop=True)


## 5. Additive spline sensitivity

The linear decomposition is deliberately interpretable. A shared-knot additive spline model
is used as a sensitivity check for curvature. Knots are fixed from an equal-simulation
weighted distribution, and the same basis and ridge penalty are used in both simulations.
All outcomes are fitted jointly. Linearity of the multi-output ridge solution therefore
preserves the pair-sum identity.

This model allows curved one-dimensional relations but not arbitrary high-order
interactions. Its adjusted gaps and conditional curves are descriptive and do not receive
bootstrap intervals.


In [ ]:
def weighted_quantile(
    values: np.ndarray,
    probabilities: np.ndarray,
    weights: np.ndarray,
) -> np.ndarray:
    order = np.argsort(values)
    ordered_values = np.asarray(values)[order]
    ordered_weights = np.asarray(weights, dtype=np.float64)[order]
    cumulative = np.cumsum(ordered_weights)
    cumulative /= cumulative[-1]
    return np.interp(probabilities, cumulative, ordered_values)


def equal_reference_weights(n_source: int, n_transfer: int) -> np.ndarray:
    return np.concatenate([
        np.full(n_source, 0.5 / n_source),
        np.full(n_transfer, 0.5 / n_transfer),
    ])


def prepare_spline_mapping(
    analysis: str,
    subset: str,
    half_width: float,
) -> dict | None:
    specification = ANALYSES[analysis]
    selected_source = select_population(source, specification, subset, half_width)
    selected_transfer = select_population(transfer, specification, subset, half_width)
    if min(len(selected_source), len(selected_transfer)) < MIN_ROWS_PER_SIMULATION:
        return None

    continuous_columns = specification["predictors"]
    combined = pd.concat([selected_source, selected_transfer], ignore_index=True)
    reference_weights = equal_reference_weights(len(selected_source), len(selected_transfer))
    knot_probabilities = np.linspace(0.0, 1.0, SPLINE_N_KNOTS)
    knots = np.column_stack([
        weighted_quantile(
            combined[column].to_numpy(np.float64),
            knot_probabilities,
            reference_weights,
        )
        for column in continuous_columns
    ])
    transformer = SplineTransformer(
        n_knots=SPLINE_N_KNOTS,
        degree=SPLINE_DEGREE,
        knots=knots,
        include_bias=False,
        extrapolation="linear",
    )
    transformer.fit(combined[continuous_columns].to_numpy(np.float64))

    def transform(frame: pd.DataFrame) -> np.ndarray:
        basis = transformer.transform(frame[continuous_columns].to_numpy(np.float64))
        if subset == "pooled":
            basis = np.column_stack([basis, frame["same_host"].to_numpy(np.float64)])
        return basis

    outcome_columns = list(specification["outcomes"].values())
    source_design = transform(selected_source)
    transfer_design = transform(selected_transfer)
    source_targets = selected_source[outcome_columns].to_numpy(np.float64)
    transfer_targets = selected_transfer[outcome_columns].to_numpy(np.float64)
    source_model = Ridge(alpha=SPLINE_RIDGE_ALPHA, fit_intercept=True).fit(
        source_design, source_targets
    )
    transfer_model = Ridge(alpha=SPLINE_RIDGE_ALPHA, fit_intercept=True).fit(
        transfer_design, transfer_targets
    )
    return {
        "source": selected_source,
        "transfer": selected_transfer,
        "outcome_names": list(specification["outcomes"]),
        "transform": transform,
        "source_design": source_design,
        "transfer_design": transfer_design,
        "source_targets": source_targets,
        "transfer_targets": transfer_targets,
        "source_model": source_model,
        "transfer_model": transfer_model,
    }


spline_summary_rows = []
curve_rows = []

for analysis, specification in ANALYSES.items():
    for subset in SUBSETS:
        for half_width in OVERLAP_HALF_WIDTHS:
            prepared = prepare_spline_mapping(analysis, subset, half_width)
            if prepared is None:
                spline_summary_rows.append({
                    "analysis": analysis,
                    "subset": subset,
                    "half_width": half_width,
                    "score_interval": interval_label(half_width),
                    "status": "insufficient_rows",
                })
                continue

            source_on_source = prepared["source_model"].predict(prepared["source_design"])
            transfer_on_source = prepared["transfer_model"].predict(prepared["source_design"])
            source_on_transfer = prepared["source_model"].predict(prepared["transfer_design"])
            transfer_on_transfer = prepared["transfer_model"].predict(prepared["transfer_design"])
            adjusted_gaps = 0.5 * (
                np.mean(transfer_on_source - source_on_source, axis=0)
                + np.mean(transfer_on_transfer - source_on_transfer, axis=0)
            )
            for outcome_index, outcome in enumerate(prepared["outcome_names"]):
                spline_summary_rows.append({
                    "analysis": analysis,
                    "feature_space": specification["feature_space"],
                    "outcome": outcome,
                    "subset": subset,
                    "half_width": half_width,
                    "score_interval": interval_label(half_width),
                    "status": "ok",
                    "n_flamingo": len(prepared["source"]),
                    "n_tng": len(prepared["transfer"]),
                    "adjusted_gap_tng_minus_flamingo": adjusted_gaps[outcome_index],
                    "rmse_flamingo": np.sqrt(np.mean(
                        (
                            source_on_source[:, outcome_index]
                            - prepared["source_targets"][:, outcome_index]
                        ) ** 2
                    )),
                    "rmse_tng": np.sqrt(np.mean(
                        (
                            transfer_on_transfer[:, outcome_index]
                            - prepared["transfer_targets"][:, outcome_index]
                        ) ** 2
                    )),
                })

            if analysis != "pair_sum_decomposition" or half_width != PRIMARY_HALF_WIDTH:
                continue
            source_reference = prepared["source"].sample(
                n=min(CURVE_REFERENCE_PER_SIMULATION, len(prepared["source"])),
                replace=False,
                random_state=RANDOM_SEED,
            )
            transfer_reference = prepared["transfer"].sample(
                n=min(CURVE_REFERENCE_PER_SIMULATION, len(prepared["transfer"])),
                replace=False,
                random_state=RANDOM_SEED + 1,
            )
            reference = pd.concat([source_reference, transfer_reference], ignore_index=True)
            reference_weights = equal_reference_weights(
                len(source_reference), len(transfer_reference)
            )
            mass_column = "log_stellar_sum"
            combined_values = np.concatenate([
                prepared["source"][mass_column].to_numpy(np.float64),
                prepared["transfer"][mass_column].to_numpy(np.float64),
            ])
            weights = equal_reference_weights(
                len(prepared["source"]), len(prepared["transfer"])
            )
            lower, upper = weighted_quantile(
                combined_values, np.array([0.05, 0.95]), weights
            )
            for grid_value in np.linspace(lower, upper, CURVE_GRID_SIZE):
                varied = reference.copy()
                varied[mass_column] = grid_value
                design = prepared["transform"](varied)
                source_prediction = prepared["source_model"].predict(design)
                transfer_prediction = prepared["transfer_model"].predict(design)
                difference = transfer_prediction - source_prediction
                for outcome_index, outcome in enumerate(prepared["outcome_names"]):
                    curve_rows.append({
                        "analysis": analysis,
                        "outcome": outcome,
                        "subset": subset,
                        "half_width": half_width,
                        "mass_term": mass_column,
                        "feature_value": grid_value,
                        "predicted_flamingo": np.average(
                            source_prediction[:, outcome_index], weights=reference_weights
                        ),
                        "predicted_tng": np.average(
                            transfer_prediction[:, outcome_index], weights=reference_weights
                        ),
                        "difference_tng_minus_flamingo": np.average(
                            difference[:, outcome_index], weights=reference_weights
                        ),
                    })

spline_summary = pd.DataFrame(spline_summary_rows)
conditional_curves = pd.DataFrame(curve_rows)
spline_summary.to_csv(OUTPUT / "decomposition_spline_summary.csv", index=False)
conditional_curves.to_csv(OUTPUT / "decomposition_conditional_curves.csv", index=False)

main_spline = spline_summary.loc[
    (spline_summary["analysis"] == "pair_sum_decomposition")
    & (spline_summary["status"] == "ok")
].pivot_table(
    index=["subset", "half_width"],
    columns="outcome",
    values="adjusted_gap_tng_minus_flamingo",
)
spline_identity_residual = (
    main_spline["stellar_to_m200c"]
    - main_spline["stellar_to_bound"]
    - main_spline["bound_to_m200c"]
)
if spline_identity_residual.abs().max() > 1e-8:
    raise RuntimeError("The spline decomposition identity failed.")

spline_summary.loc[
    (spline_summary["half_width"] == PRIMARY_HALF_WIDTH)
    & (spline_summary["status"] == "ok"),
    [
        "analysis", "outcome", "subset", "n_flamingo", "n_tng",
        "adjusted_gap_tng_minus_flamingo", "rmse_flamingo", "rmse_tng",
    ],
].round(4).reset_index(drop=True)


## 6. Figures

The first figure shows how the two stage gaps and their total vary with the score interval.
The second shows component-bootstrap intervals for the primary adjusted gaps. The third
shows the spline decomposition across the central 90% of the equal-simulation stellar-sum
reference distribution.

The total is not an independent fourth measurement: it is the sum of the two stages. Its
display provides an identity check and connects this notebook to the previously observed
stellar-sum-to-M200c discrepancy.


In [ ]:
OUTCOME_LABELS = {
    "stellar_to_bound": "stellar to bound",
    "bound_to_m200c": "bound to M200c",
    "stellar_to_m200c": "stellar to M200c (total)",
    "primary_bound_per_stellar": "primary bound per stellar",
    "secondary_bound_per_stellar": "secondary bound per stellar",
}
OUTCOME_COLOURS = {
    "stellar_to_bound": "tab:blue",
    "bound_to_m200c": "tab:orange",
    "stellar_to_m200c": "tab:purple",
}


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, subset in zip(axes, SUBSETS):
    current = linear_summary.loc[
        (linear_summary["analysis"] == "pair_sum_decomposition")
        & (linear_summary["subset"] == subset)
        & (linear_summary["status"] == "ok")
    ]
    for outcome in MAIN_OUTCOMES:
        line = current.loc[current["outcome"] == outcome].sort_values("half_width")
        ax.plot(
            line["half_width"],
            line["adjusted_gap_tng_minus_flamingo"],
            marker="o",
            color=OUTCOME_COLOURS[outcome],
            label=OUTCOME_LABELS[outcome],
        )
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
    ax.set_title(subset.replace("_", " ").title())
    ax.set_xlabel("Overlap half-width about 0.5")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Adjusted TNG minus FLAMINGO gap [dex]")
axes[-1].legend(loc="best", fontsize=8)
fig.suptitle("Linear mass-chain decomposition", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT / "decomposition_linear_gap_sensitivity.png", dpi=220, bbox_inches="tight")
fig.savefig(OUTPUT / "decomposition_linear_gap_sensitivity.pdf", bbox_inches="tight")
plt.show()


current = bootstrap_summary.loc[
    bootstrap_summary["estimand"] == "adjusted_gap"
].copy()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, analysis in enumerate(ANALYSES):
    for column, subset in enumerate(SUBSETS):
        ax = axes[row, column]
        selected = current.loc[
            (current["analysis"] == analysis) & (current["subset"] == subset)
        ].reset_index(drop=True)
        positions = np.arange(len(selected))
        ax.hlines(
            positions, selected["lower_95"], selected["upper_95"],
            color="tab:blue", linewidth=1.5,
        )
        ax.plot(selected["estimate"], positions, "o", color="tab:blue")
        ax.axvline(0.0, color="black", linestyle="--", linewidth=1)
        ax.set_yticks(
            positions,
            [OUTCOME_LABELS[name] for name in selected["outcome"]],
            fontsize=8,
        )
        if row == 0:
            ax.set_title(subset.replace("_", " ").title())
        if column == 0:
            ax.set_ylabel(analysis.replace("_", " ").title())
        if row == 1:
            ax.set_xlabel("Adjusted TNG minus FLAMINGO gap [dex]")
        ax.grid(axis="x", alpha=0.25)
fig.suptitle("Connected-component bootstrap intervals inside 0.4--0.6", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT / "decomposition_component_intervals.png", dpi=220, bbox_inches="tight")
fig.savefig(OUTPUT / "decomposition_component_intervals.pdf", bbox_inches="tight")
plt.show()


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, subset in zip(axes, SUBSETS):
    current = conditional_curves.loc[conditional_curves["subset"] == subset]
    for outcome in MAIN_OUTCOMES:
        line = current.loc[current["outcome"] == outcome]
        ax.plot(
            line["feature_value"],
            line["difference_tng_minus_flamingo"],
            color=OUTCOME_COLOURS[outcome],
            linewidth=2,
            label=OUTCOME_LABELS[outcome],
        )
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
    ax.set_title(subset.replace("_", " ").title())
    ax.set_xlabel("log10 combined stellar mass")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Predicted TNG minus FLAMINGO difference [dex]")
axes[-1].legend(loc="best", fontsize=8)
fig.suptitle("Spline mass-chain decomposition inside 0.4--0.6", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT / "decomposition_spline_curves.png", dpi=220, bbox_inches="tight")
fig.savefig(OUTPUT / "decomposition_spline_curves.pdf", bbox_inches="tight")
plt.show()


## 7. Interpretation rules

1. The first-stage and second-stage gaps must be interpreted together. A larger magnitude
 identifies where most of the fitted total discrepancy enters under this decomposition;
 it does not prove a unique physical mechanism.
2. A stage interval excluding zero supports a persistent conditional difference under the
 chosen linear form and fixed overlap selection. An interval containing zero indicates
 that the stage is unresolved, not that the mappings are identical.
3. Agreement between the linear and spline signs strengthens the localisation. Strong
 curvature or a sign change across stellar mass should be reported rather than compressed
 into one mean gap.
4. The member-level results test whether the first stage is concentrated in the
 stellar-primary or stellar-secondary member. They remain supporting evidence because
 pair membership and bound-mass assignment can inherit catalogue definitions.
5. Same-host estimates are generally better supported numerically. Different-host results
 remain exploratory when retained TNG counts are small.
6. No multiplicity correction is applied across stages, host subsets or sensitivity checks.
 Conclusions should therefore emphasise coherent patterns and uncertainty, not isolated
 threshold crossings.

If the first stage dominates, the next diagnostic should inspect stellar-to-bound mass
conversion and member asymmetry. If the second stage dominates, the next diagnostic should
inspect the relation between pair-bound mass, host assignment and M200c. If both contribute,
both catalogue levels should be treated as part of the transfer shift before any targeted
posterior retraining is considered.


## 8. Output manifest

The manifest records the fixed input result, population definitions, decomposition identity,
regression settings, software versions and exported files. It also records that this
notebook performs no posterior training or updating.


In [ ]:
output_manifest = {
    "experiment_id": f"mass_mapping_decomposition_{timestamp}",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project": str(PROJECT),
    "git_revision": git_revision,
    "output": str(OUTPUT),
    "purpose": (
        "Decompose the conditional stellar-sum-to-M200c simulation gap into "
        "stellar-to-bound and bound-to-M200c stages; no NPE training"
    ),
    "notebook": str(
        PROJECT / "notebooks" / "diagnostics" / "mass_mapping_decomposition.ipynb"
    ),
    "domain_result_id": DOMAIN_RESULT_ID,
    "domain_manifest_sha256": sha256(DOMAIN_MANIFEST),
    "versions": versions,
    "settings": {
        "random_seed": RANDOM_SEED,
        "bootstrap_repeats": BOOTSTRAP_REPEATS,
        "overlap_half_widths": OVERLAP_HALF_WIDTHS,
        "primary_half_width": PRIMARY_HALF_WIDTH,
        "minimum_rows_per_simulation": MIN_ROWS_PER_SIMULATION,
        "linear_reference": "equal simulation weight",
        "bootstrap_unit": "connected host component within simulation",
        "bootstrap_fixed_quantities": [
            "selected population", "domain scores", "reference centre",
            "reference scale", "linear specification",
        ],
        "spline_n_knots": SPLINE_N_KNOTS,
        "spline_degree": SPLINE_DEGREE,
        "spline_ridge_alpha": SPLINE_RIDGE_ALPHA,
        "curve_grid_size": CURVE_GRID_SIZE,
        "curve_reference_per_simulation": CURVE_REFERENCE_PER_SIMULATION,
    },
    "analyses": ANALYSES,
    "mass_chain_identity": (
        "log_m200c_minus_log_stellar_sum = "
        "log_bound_subhalo_sum_minus_log_stellar_sum + "
        "log_m200c_minus_log_bound_subhalo_sum"
    ),
    "maximum_linear_identity_residual": float(
        identity_audit["identity_residual"].abs().max()
    ),
    "maximum_bootstrap_identity_residual": float(max(bootstrap_identity_residuals)),
    "maximum_spline_identity_residual": float(spline_identity_residual.abs().max()),
    "source_population": "fixed 21,409-pair held-out FLAMINGO population",
    "transfer_population": "all 1,530 corrected TNG pairs",
    "reporting_notes": [
        "Primary selection uses the fixed stellar-sum out-of-fold domain score.",
        "Adjusted gaps compare fitted outcomes at an equal-simulation reference centre.",
        "Member bound masses are aligned using the stellar-primary endpoint flag.",
        "Different-host results are exploratory when retained counts are small.",
        "Intervals exclude retraining, classifier uncertainty and cosmic variance.",
        "No NPE was fitted, updated or selected in this notebook.",
    ],
    "outputs": sorted(
        str(path.relative_to(OUTPUT))
        for path in OUTPUT.rglob("*")
        if path.is_file()
    ),
}
(OUTPUT / "mass_mapping_decomposition_manifest.json").write_text(
    json.dumps(output_manifest, indent=2), encoding="utf-8"
)

print(f"Completed mass-mapping decomposition export: {OUTPUT}")
print("No NPE was trained or updated.")
